# 00 · Ambiente, dati e protocollo

Tesi: *Valutazione di una strategia di market making mediante dati limit order book L2, L3 e L4*.

Questo notebook fissa l'ambiente, verifica i dati grezzi contro il manifesto (URL, data di accesso, SHA-256), mostra le configurazioni e la suddivisione cronologica delle sessioni, ed esegue i test automatici. I notebook successivi leggono i risultati prodotti dalla pipeline:

```powershell
.venv/Scripts/python.exe scripts/download_data.py --fi2010-zip <zip Etsin>
.venv/Scripts/mm-backtest.exe ingest      # grezzi -> Parquet in data/interim
.venv/Scripts/mm-backtest.exe pipeline    # qualità, calibrazione, validazione, test, stress
```

| Componente del piano | Modulo | Output |
|---|---|---|
| parsers | `parsers/` (FI-2010, ITCH, Hyperliquid), `ingest.py`, `storage.py` | eventi comuni in Parquet |
| orderbook | `orderbook.py` | stato L2 o MBO |
| strategy | `strategy.py` | intenti di ordine |
| execution | `execution.py` (touch, price-through, cross, coda FIFO, latenza) | fill simulati |
| accounting | `accounting.py` | cassa, inventario, fee, mark to market |
| evaluation | `evaluation.py`, `pipeline.py` | metriche, tabelle, confronti |

| Esperimento | Dataset | Modello di fill | Notebook |
|---|---|---|---|
| E1 | FI-2010 | touch | 02 |
| E2 | FI-2010 | price-through (e `cross`, baseline aggiuntiva) | 02 |
| E3 | Nasdaq ITCH | coda FIFO, con vista L2 dallo stesso feed | 04 |
| E4 | Hyperliquid | coda FIFO L4 | 05 |
| E5 | ITCH e Hyperliquid | coda FIFO con stress di latenza e costi | 06 |

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidates if (p / "src/market_making_tesi").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from market_making_tesi import report as R
from market_making_tesi.report import it, pct
R.setup_plots()
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
np.random.seed(0)
import platform
from importlib.metadata import version
environment = {"python": platform.python_version(), "platform": platform.platform(),
               **{n: version(n) for n in ["numpy", "pandas", "pyarrow", "matplotlib", "psutil", "nbformat", "nbclient"]}}
(ROOT / "results" / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
display(pd.Series(environment, name="versione").to_frame())

,versione
python,3.14.6
platform,Windows-11-10.0.26300-SP0
numpy,2.5.4
pandas,3.0.6
pyarrow,26.0.0
matplotlib,3.11.2
psutil,7.2.2
nbformat,5.11.1
nbclient,0.11.0


## Dati grezzi e manifesto

Ogni file grezzo è registrato in `data/manifest.json` con fonte, data di accesso, dimensione e SHA-256. La cella ricalcola gli hash (alcuni minuti: circa 40 GB complessivi, quasi tutti ITCH).

In [2]:
import hashlib
VERIFICA_HASH = True
manifest = json.loads((ROOT / "data" / "manifest.json").read_text(encoding="utf-8"))
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            digest.update(block)
    return digest.hexdigest()
rows = []
for key, entry in sorted(manifest.items()):
    path = ROOT / key
    ok = path.exists() and path.stat().st_size == entry["bytes"]
    if ok and VERIFICA_HASH:
        ok = sha256(path) == entry["sha256"]
    rows.append({"file": key.removeprefix("data/raw/"), "MB": round(entry["bytes"] / 1e6, 1),
                 "accesso": entry["accessed"], "fonte": entry["source"].split("/")[2], "verificato": ok})
files = pd.DataFrame(rows)
display(files)
assert files.verificato.all(), "file mancanti o alterati: rieseguire scripts/download_data.py"
print(f"{len(files)} file, {it(files.MB.sum() / 1000, 1)} GB in totale")

,file,MB,accesso,fonte,verificato
0,fi2010/fi2010_day01.txt,94.2,2026-10-14,etsin.fairdata.fi,True
1,fi2010/fi2010_day02.txt,91.5,2026-10-14,etsin.fairdata.fi,True
2,fi2010/fi2010_day03.txt,68.0,2026-10-14,etsin.fairdata.fi,True
3,fi2010/fi2010_day04.txt,88.3,2026-10-14,etsin.fairdata.fi,True
4,fi2010/fi2010_day05.txt,82.9,2026-10-14,etsin.fairdata.fi,True
5,fi2010/fi2010_day06.txt,93.3,2026-10-14,etsin.fairdata.fi,True
6,fi2010/fi2010_day07.txt,89.0,2026-10-14,etsin.fairdata.fi,True
7,fi2010/fi2010_day08.txt,132.3,2026-10-14,etsin.fairdata.fi,True
8,fi2010/fi2010_day09.txt,124.4,2026-10-14,etsin.fairdata.fi,True
9,fi2010/fi2010_day10.txt,76.1,2026-10-14,etsin.fairdata.fi,True


29 file, 32,7 GB in totale


## Configurazioni e protocollo temporale

La regola operativa è la stessa per tutti i modelli di fill. **La strategia opera solo a bassa volatilità**: a ogni record stima la volatilità come deviazione standard dei rendimenti del midpoint su una finestra mobile e quota solo se la stima è sotto una soglia. La finestra è di *W* record per FI-2010 (campionamento fisso) e Hyperliquid (senza timestamp), di *W* secondi per ITCH (midpoint campionato ogni secondo, così la stima non dipende dall'intensità dei messaggi); sopra soglia cancella, liquida e resta ferma finché la volatilità non rientra.

La suddivisione è cronologica, come nel piano: la prima sessione di ogni strumento è di **calibrazione** (la soglia è il quantile *q* della volatilità osservata in quella sessione, per ciascuna finestra), la successiva di **validazione** (si sceglie la combinazione di finestra *W*, quantile *q* (50%, 75% o 90%: si opera sotto la mediana, il terzo quartile o il 90° percentile) e durata degli ordini; le sessioni successive formano il **test set**, mai usato per scegliere soglie o parametri. Lo stadio di **ablazione** ripete il test con la strategia sempre attiva, per misurare il contributo del filtro.

In [3]:
from dataclasses import asdict
from market_making_tesi.config import StrategyConfig
from market_making_tesi.pipeline import DATASETS, session_list
configs = {n: asdict(StrategyConfig.from_json(ROOT / "config" / f"{n}.json")) for n in ["fi2010", "itch", "hyperliquid"]}
display(pd.DataFrame(configs))
rows = []
for name, spec in DATASETS.items():
    for session, symbol, _ in session_list(ROOT, spec):
        rows.append({"dataset": R.DATASET_LABELS[name], "sessione": session, "simbolo": symbol,
                     "insieme": "calibrazione" if session in spec.calibration
                                else "validazione" if session in spec.validation else "test"})
sessions = pd.DataFrame(rows)
display(sessions.groupby(["dataset", "insieme"]).agg(sessioni=("sessione", "size"),
        dalla=("sessione", "min"), alla=("sessione", "max"), simboli=("simbolo", "nunique")))
display(pd.DataFrame({n: {"griglia": s.grid or "da ITCH", "modello di riferimento": s.reference,
                          "orizzonte markout (record)": s.markout_records,
                          "scenari di costo": ", ".join(s.fees)} for n, s in DATASETS.items()}).T)

,fi2010,itch,hyperliquid
quote_size,1.0,100.0,0.01
volatility_window,50,120,500
volatility_min_periods,20,20,100
volatility_threshold,1.0,1.0,1.0
volatility_method,rolling,rolling,rolling
volatility_sample_ns,None,1000000000,None
return_type,diff,log,log
ewma_lambda,0.94,0.94,0.94
max_order_age_events,100,1000000000,5000
max_order_age_ns,None,None,None


sessioni        dalla         alla  simboli
dataset         insieme                                                  
FI-2010         calibrazione         5        day01        day01        5
                test                35        day04        day10        5
                validazione         10        day02        day03        5
Hyperliquid BTC calibrazione         1  20251201T08  20251201T08        1
                test                 4  20251202T08  20251203T14        1
                validazione          1  20251201T14  20251201T14        1
Nasdaq ITCH     calibrazione         3   2019-01-30   2019-01-30        3
                test                15   2019-07-30   2020-01-30        3
                validazione          3   2019-03-27   2019-03-27        3

,griglia,modello di riferimento,orizzonte markout (record),scenari di costo
fi2010,"{'volatility_window': [20, 100], 'vol_quantile...",price-through,10,lordo
itch,"{'volatility_window': [30, 120], 'vol_quantile...",queue,100,"lordo, nasdaq_base, senza_rebate"
hyperliquid,"{'volatility_window': [500, 5000], 'vol_quanti...",queue,1000,"lordo, tier0, tier_alto"


## Test automatici e controlli sui campioni

La suite copre parser, book, regole di fill, coda FIFO, latenza, contabilità, scomposizione del P&L, metriche, Parquet e selezione dei parametri. I campioni piccoli in `data/samples` verificano il motore su sequenze con risultato noto.

In [4]:
import subprocess
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                       cwd=ROOT, capture_output=True, text=True, timeout=600)
log = check.stdout + check.stderr
(ROOT / "results" / "test_report.txt").write_text(log, encoding="utf-8")
print(log.strip().splitlines()[-3:])
assert check.returncode == 0, log
from market_making_tesi.experiments import run_samples
samples, _ = run_samples(ROOT, ROOT / "results" / "samples")
display(samples)
assert samples.fill_maker.tolist() == [22, 5, 2, 2, 0]

['Ran 47 tests in 0.271s', '', '\x1bOK\x1b']


,esperimento,granularita,record,ordini_inseriti,ordini_cancellati,fill_maker,fill_taker,fill_totali,equity_finale,commissioni
0,FI-2010 touch,L2,60,24,2,22,2,24,0.0108,0.0
1,FI-2010 price-through,L2,60,8,3,5,3,8,-0.0001,0.0
2,Nasdaq ITCH controllato,L3,10,4,2,2,0,2,0.0100,0.0
3,Hyperliquid controllato,L4,10,4,2,2,0,2,0.1000,0.0
4,Hyperliquid reale book diff (non valutabile),L4 senza trade,500,50,50,0,0,0,0.0000,0.0
